In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q12/annotated/checkpoints/pre_cell_2.pickle

trying: ['tpch_parent']
me:  0
trying: ['load_orders']
me:  2
trying: ['load_lineitem']
me:  1
trying: ['pd']
me:  0
trying: ['STORAGE_OPTS']
me:  0
trying: ['lineitem']


me:  1
trying: ['DATA_ROOT']
me:  0
trying: ['orders']


me:  2


Declaring variable tpch_parent
Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable DATA_ROOT
Declaring variable load_lineitem
Declaring variable lineitem
Declaring variable load_orders
Declaring variable orders


In [4]:
%%RecordEvent
%%time
### cell 2 ###

### cell 2 (optimized for cudf) ###

# filter bounds (kept on GPU)
date1 = pd.Timestamp("1994-01-01")
date2 = pd.Timestamp("1995-01-01")

# 1) filter lineitem and project only the columns needed for the join and grouping
flineitem = (
    lineitem
    [(lineitem.L_RECEIPTDATE >= date1)
     & (lineitem.L_RECEIPTDATE < date2)
     & (lineitem.L_COMMITDATE   < date2)
     & (lineitem.L_SHIPDATE      < date2)
     & (lineitem.L_SHIPDATE      < lineitem.L_COMMITDATE)
     & (lineitem.L_COMMITDATE    < lineitem.L_RECEIPTDATE)
     & lineitem.L_SHIPMODE.isin(["MAIL", "SHIP"]) ]
    [["L_ORDERKEY", "L_SHIPMODE"]]
)

# 2) project only the columns we need from orders
small_orders = orders[["O_ORDERKEY", "O_ORDERPRIORITY"]]

# 3) join on the minimal set of columns
jn = flineitem.merge(
    small_orders,
    left_on="L_ORDERKEY",
    right_on="O_ORDERKEY"
)

# 4) build the urgency flags (all on GPU)
jn["is_urgent"]     = jn.O_ORDERPRIORITY.isin(["1-URGENT", "2-HIGH"])
jn["is_not_urgent"] = ~jn["is_urgent"]

# 5) group and aggregate in one pass
total = jn.groupby("L_SHIPMODE", as_index=False).agg(
    g1=("is_urgent",     "sum"),
    g2=("is_not_urgent", "sum")
)

print(total)

  L_SHIPMODE    g1    g2
0       MAIL  6202  9324
1       SHIP  6200  9262
CPU times: user 233 ms, sys: 128 ms, total: 362 ms
Wall time: 368 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q12/rewritten/o4_mini_high_q12/checkpoints/post_cell_2_try_1.pickle

migration speed (bps): 788158979.7627001
---------------------------
variables to migrate:
jn 48
date2 48
DATA_ROOT 80
orders 48
tpch_parent 54
load_lineitem 144
STORAGE_OPTS 64
date1 48
lineitem 48
total 48
load_orders 144
small_orders 48
flineitem 48
pd 72
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q12/rewritten/o4_mini_high_q12/checkpoints/post_cell_2_try_1.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['orders']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['date1', 'small_orders', 'total', 'date2', 'jn', 'flineitem']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q12/opt_cell_exec_info_2_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[2], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q12/annotated/checkpoints/post_cell_2.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
